# Stage 4 - Silver Cleaning

Reads `workspace.default.capstone_bronze_sales`, applies:

- Data-type casting
- Null handling & trimming
- Business validation (removes INVALID records)
- Derived date columns: `year`, `month`, `month_name`

Writes clean data to `workspace.default.capstone_silver_sales`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

spark = SparkSession.builder.getOrCreate()

## 1. Configuration

In [ ]:
BRONZE_TABLE = "workspace.default.capstone_bronze_sales"
SILVER_TABLE = "workspace.default.capstone_silver_sales"

## 2. Load Bronze

In [ ]:
df_bronze = spark.read.table(BRONZE_TABLE)
print(f"Bronze row count: {df_bronze.count()}")

## 3. Data-type casting

In [ ]:
df_typed = (
    df_bronze
    # String columns - trim whitespace
    .withColumn("order_id",       F.trim(F.col("order_id").cast(StringType())))
    .withColumn("customer_id",    F.trim(F.col("customer_id").cast(StringType())))
    .withColumn("customer_name",  F.trim(F.col("customer_name").cast(StringType())))
    .withColumn("city",           F.trim(F.col("city").cast(StringType())))
    .withColumn("state",          F.trim(F.col("state").cast(StringType())))
    .withColumn("product_id",     F.trim(F.col("product_id").cast(StringType())))
    .withColumn("product_name",   F.trim(F.col("product_name").cast(StringType())))
    .withColumn("category",       F.trim(F.col("category").cast(StringType())))
    .withColumn("payment_method", F.trim(F.col("payment_method").cast(StringType())))
    .withColumn("order_status",   F.trim(F.col("order_status").cast(StringType())))
    # Numeric columns
    .withColumn("quantity",        F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price",      F.col("unit_price").cast(DoubleType()))
    .withColumn("discount_pct",    F.col("discount_pct").cast(DoubleType()))
    .withColumn("gross_amount",    F.col("gross_amount").cast(DoubleType()))
    .withColumn("discount_amount", F.col("discount_amount").cast(DoubleType()))
    .withColumn("net_amount",      F.col("net_amount").cast(DoubleType()))
    # Date column
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
)

## 4. Null handling - fill safe defaults

In [ ]:
df_nullfilled = (
    df_typed
    .withColumn("discount_pct",    F.coalesce(F.col("discount_pct"),    F.lit(0.0)))
    .withColumn("discount_amount", F.coalesce(F.col("discount_amount"), F.lit(0.0)))
    .withColumn("city",            F.coalesce(F.col("city"),            F.lit("Unknown")))
    .withColumn("state",           F.coalesce(F.col("state"),           F.lit("Unknown")))
    .withColumn("payment_method",  F.coalesce(F.col("payment_method"),  F.lit("Unknown")))
)

## 5. Remove invalid business records

In [ ]:
df_valid = df_nullfilled.filter(F.col("quality_flag") == "VALID")

removed = df_nullfilled.count() - df_valid.count()
print(f"Records removed (invalid): {removed}")
print(f"Valid records retained   : {df_valid.count()}")

## 6. Add derived date columns

In [ ]:
df_silver = (
    df_valid
    .withColumn("year",       F.year(F.col("order_date")))
    .withColumn("month",      F.month(F.col("order_date")))
    .withColumn("month_name", F.date_format(F.col("order_date"), "MMMM"))
)

## 7. Write Silver table

In [ ]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

count = spark.read.table(SILVER_TABLE).count()
print(f"Silver table written: {SILVER_TABLE}")
print(f"Row count           : {count}")

## 8. Quick validation

In [ ]:
print("=== Silver schema ===")
spark.read.table(SILVER_TABLE).printSchema()

print("=== Sample rows ===")
spark.read.table(SILVER_TABLE).show(5, truncate=False)

print("=== Null check on key columns ===")
silver_df = spark.read.table(SILVER_TABLE)
for col_name in ["order_id", "customer_id", "product_id", "quantity", "net_amount", "order_date"]:
    null_count = silver_df.filter(F.col(col_name).isNull()).count()
    print(f"  {col_name}: {null_count} nulls")